In [1]:
##### KNN #####
# K-Nearest Neighbours
# As we already know, our data is represented by points in an n_feature dimensional space. If you have 3 features, its R3 etc.
# this algorithm works by taking the first point, placing it and knowing its outcome. afterwards, when placing the next point and checks how close it is
# if its closer to a point with a different outcome, then the prediction will be that the newly placed point belongs to the other category
# example : lets take a 2d space
#
#      ^
#      |               x2
#      |
#      |     x1
#      | - -- ---  -- >
#
# now we want to place x3. we saved x1 and x2 as train data and we know they have different outcomes. when placing x3 we will check to what point its closer to. if its x2, we know that x3 will likely hold the same value, in this case the patient will either have B or M
# now this can be innacurate. so how do we solve it? we take majority vote. if instead of x2 we had 200 points and the majority is an answer, then the same answer will be predicted for our newly placed point
import numpy as np
import pandas as pd
import kagglehub
from collections import Counter
from sklearn.preprocessing import StandardScaler

def euclidean_distance(start, finish):
    return (np.sum( (finish - start)**2 ))**0.5

def manhattan_distance(start, finish):
    return np.sum(abs(finish - start))

def select_decision(start, finish, type=0):
    if type == 0:
        return euclidean_distance(start, finish)
    else:
        return manhattan_distance(start, finish)


x, y = None, None
dataset, model = None, None
decision = None

#decision = int(input("Choose how to compute distance -> "))
def load_data():
    """
    Preprocessing function
    """
    global x, y
    global dataset

    dataset = pd.read_csv("/kaggle/input/datasets/organizations/uciml/breast-cancer-wisconsin-data/data.csv")
    dataset["diagnosis"] = dataset["diagnosis"].replace({"M" : 1, "B" : 0})
    y = np.array(dataset["diagnosis"])
    dataset = dataset.drop(columns=["id", "diagnosis", "Unnamed: 32"])
    x = np.array(dataset)

class KNN():
    """
    K-Nearest Neighbours class. Look at the points already placed as training and "place" the newest one
    to the K closest. By the verb place one must understand that we take majority vote to determine the label.
    """
    def __init__(self, k=5):
        self.knnval = k

    def fit(self, x, y):
        self.tx = x
        self.ty = y

    def _pred(self, sample):
        distances = []
        for alfa in self.tx: # compare it to the other points that have been "placed", we find the k nearest points distance wise
            dist = euclidean_distance(sample, alfa)
            distances.append(dist)

        indices = np.argsort(distances)[:self.knnval] # we take the first k
        results = [self.ty[idx] for idx in indices ] #we take the values they hold(malign or benign in our case)

        return Counter(results).most_common(1)[0][0] # we do majority
        
    def predict(self, x):
        preds = []
        for sample in x: # for each example we "place it" in an imaginary n dimensional graph, n = nb_features
            temp = self._pred(sample)
            preds.append(temp)

        return np.array(preds)
        
load_data()
x_train, x_test = x[:int(len(x) * 0.8)], x[int(len(x) * 0.8):]
y_train, y_test = y[:int(len(y) * 0.8)], y[int(len(y) * 0.8):]
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

model = KNN()
model.fit(x_train, y_train)
predict = model.predict(x_test)
count = 0
for idx, elem in enumerate(predict):
    if elem == y_test[idx]:
        count = count+1

print(f"Precision -> {count/len(predict)* 100:.4f} %")
##### Use cases #####
# When we are dealing with a small quantity of data that is fairly easy to distinguish
# A lot of features as this helps ease the process of splitting data
# This is also used in famous libraries like ChromaDB! It's also a part of RAG, before sending the LLM the documents
# one can use KNN to determine the most relevant docs/ the appropriate context

/tmp/ipykernel_58/1764649906.py:39: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset["diagnosis"] = dataset["diagnosis"].replace({"M" : 1, "B" : 0})


Precision -> 96.4912 %
